# Sales & Demand Forecasting System
## Building Predictive Models for Business Planning

This notebook demonstrates a complete sales forecasting workflow including:
- Data exploration and cleaning
- Time-series feature engineering
- Multiple forecasting approaches
- Model evaluation and selection
- Business-ready visualizations

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import timedelta
import os

warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print('✅ All libraries imported successfully!')

## 1. Data Loading and Exploration

Let's start by loading and understanding our sales data.

In [ ]:
# Load the dataset
df = pd.read_csv('../data/sales_data.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 10 rows:")
print(df.head(10))
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## 2. Data Preparation and Cleaning

In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

# For forecasting, we'll aggregate sales by date (sum across all stores and categories)
daily_sales = df.groupby('date')['sales'].sum().reset_index()
daily_sales.columns = ['date', 'sales']

print(f"Aggregated data shape: {daily_sales.shape}")
print(f"Date range: {daily_sales['date'].min()} to {daily_sales['date'].max()}")
print(f"\nDaily sales statistics:")
print(daily_sales['sales'].describe())
print(f"\nNo missing values: {daily_sales.isnull().sum().sum() == 0}")

## 3. Time-Series Visualization

In [ ]:
# Visualize the time series
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Full time series
axes[0].plot(daily_sales['date'], daily_sales['sales'], linewidth=2, color='#2E86AB')
axes[0].set_title('Daily Sales Over Time - Full Period', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Sales ($)')
axes[0].grid(True, alpha=0.3)

# Monthly aggregated sales (trend visualization)
monthly_sales = daily_sales.copy()
monthly_sales['year_month'] = monthly_sales['date'].dt.to_period('M')
monthly_agg = monthly_sales.groupby('year_month')['sales'].sum().reset_index()
monthly_agg['year_month'] = monthly_agg['year_month'].astype(str)

axes[1].bar(range(len(monthly_agg)), monthly_agg['sales'], color='#A23B72', alpha=0.8)
axes[1].set_title('Monthly Total Sales - Identifying Seasonality', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Sales ($)')
axes[1].set_xticks(range(0, len(monthly_agg), 3))
axes[1].set_xticklabels(monthly_agg['year_month'][::3], rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../output/01_time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Time series visualization saved!")

## 4. Feature Engineering for Time-Series Forecasting

In [ ]:
# Create a copy for feature engineering
df_features = daily_sales.copy()

# Extract time-based features
df_features['day'] = df_features['date'].dt.day
df_features['month'] = df_features['date'].dt.month
df_features['quarter'] = df_features['date'].dt.quarter
df_features['year'] = df_features['date'].dt.year
df_features['day_of_week'] = df_features['date'].dt.dayofweek
df_features['day_of_year'] = df_features['date'].dt.dayofyear
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)

# Create lag features (previous day's sales, previous week's sales, etc.)
df_features['lag_1'] = df_features['sales'].shift(1)  # Previous day
df_features['lag_7'] = df_features['sales'].shift(7)  # Previous week
df_features['lag_30'] = df_features['sales'].shift(30)  # Previous month

# Create rolling average features
df_features['rolling_mean_7'] = df_features['sales'].rolling(window=7).mean()  # 7-day average
df_features['rolling_mean_30'] = df_features['sales'].rolling(window=30).mean()  # 30-day average

# Drop rows with NaN values (created by lag and rolling features)
df_features = df_features.dropna().reset_index(drop=True)

print(f"Features created! Shape: {df_features.shape}")
print(f"\nFeature columns: {df_features.columns.tolist()}")
print(f"\nSample data with features:")
print(df_features.head())

## 5. Train-Test Split (Time-Series Aware)

In [ ]:
# For time-series, we use temporal split (not random split)
# Train: 70%, Validation: 15%, Test: 15%

train_size = int(len(df_features) * 0.70)
val_size = int(len(df_features) * 0.15)
test_size = len(df_features) - train_size - val_size

train_data = df_features[:train_size].copy()
val_data = df_features[train_size:train_size+val_size].copy()
test_data = df_features[train_size+val_size:].copy()

print(f"Train set: {len(train_data)} samples ({train_size/len(df_features)*100:.1f}%)")
print(f"Validation set: {len(val_data)} samples ({val_size/len(df_features)*100:.1f}%)")
print(f"Test set: {len(test_data)} samples ({test_size/len(df_features)*100:.1f}%)")

# Prepare features and targets
feature_cols = ['day', 'month', 'quarter', 'year', 'day_of_week', 'day_of_year', 
                'is_weekend', 'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 'rolling_mean_30']

X_train = train_data[feature_cols]
y_train = train_data['sales']

X_val = val_data[feature_cols]
y_val = val_data['sales']

X_test = test_data[feature_cols]
y_test = test_data['sales']

print(f"\nTraining features shape: {X_train.shape}")
print(f"Training targets shape: {y_train.shape}")

## 6. Train Multiple Forecasting Models

In [ ]:
# Dictionary to store all models and results
models = {}
results = {}

# Model 1: Linear Regression
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred_val = lr_model.predict(X_val)
lr_pred_test = lr_model.predict(X_test)
models['Linear Regression'] = lr_model
results['Linear Regression'] = {
    'val_pred': lr_pred_val,
    'test_pred': lr_pred_test
}
print("✅ Linear Regression trained")

# Model 2: Random Forest
print("\nTraining Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred_val = rf_model.predict(X_val)
rf_pred_test = rf_model.predict(X_test)
models['Random Forest'] = rf_model
results['Random Forest'] = {
    'val_pred': rf_pred_val,
    'test_pred': rf_pred_test
}
print("✅ Random Forest trained")

# Model 3: Gradient Boosting
print("\nTraining Gradient Boosting...")
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, 
                                      max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)
gb_pred_val = gb_model.predict(X_val)
gb_pred_test = gb_model.predict(X_test)
models['Gradient Boosting'] = gb_model
results['Gradient Boosting'] = {
    'val_pred': gb_pred_val,
    'test_pred': gb_pred_test
}
print("✅ Gradient Boosting trained")

print("\n✅ All models trained successfully!")

## 7. Model Evaluation and Comparison

In [ ]:
# Evaluate all models
evaluation_results = []

for model_name, model_results in results.items():
    # Validation metrics
    val_mse = mean_squared_error(y_val, model_results['val_pred'])
    val_mae = mean_absolute_error(y_val, model_results['val_pred'])
    val_rmse = np.sqrt(val_mse)
    val_r2 = r2_score(y_val, model_results['val_pred'])
    
    # Test metrics
    test_mse = mean_squared_error(y_test, model_results['test_pred'])
    test_mae = mean_absolute_error(y_test, model_results['test_pred'])
    test_rmse = np.sqrt(test_mse)
    test_r2 = r2_score(y_test, model_results['test_pred'])
    
    evaluation_results.append({
        'Model': model_name,
        'Val RMSE': val_rmse,
        'Val MAE': val_mae,
        'Val R²': val_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R²': test_r2
    })

eval_df = pd.DataFrame(evaluation_results)
print("\n📊 MODEL EVALUATION RESULTS")
print("="*80)
print(eval_df.to_string(index=False))

# Find best model
best_model_name = eval_df.loc[eval_df['Test R²'].idxmax(), 'Model']
best_r2 = eval_df.loc[eval_df['Test R²'].idxmax(), 'Test R²']
print(f"\n🏆 Best Model: {best_model_name} (R² = {best_r2:.4f})")

## 8. Visualization of Model Performance

In [ ]:
# Compare models on test set
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Get test data dates for x-axis
test_dates = test_data['date'].values
test_actual = y_test.values

# Plot 1: All models comparison
ax = axes[0, 0]
ax.plot(test_dates, test_actual, 'k-', linewidth=2.5, label='Actual', zorder=3)
colors = ['#2E86AB', '#A23B72', '#F18F01']
for (model_name, model_results), color in zip(results.items(), colors):
    ax.plot(test_dates, model_results['test_pred'], '--', linewidth=2, label=model_name, color=color, alpha=0.8)
ax.set_title('Test Set: Actual vs Predicted Sales', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Sales ($)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: R² Score Comparison
ax = axes[0, 1]
bars = ax.bar(eval_df['Model'], eval_df['Test R²'], color=colors, alpha=0.8)
ax.set_title('R² Score Comparison (Test Set)', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Score')
ax.set_ylim([0, 1])
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: RMSE Comparison
ax = axes[1, 0]
bars = ax.bar(eval_df['Model'], eval_df['Test RMSE'], color=colors, alpha=0.8)
ax.set_title('Root Mean Squared Error (Test Set)', fontsize=12, fontweight='bold')
ax.set_ylabel('RMSE ($)')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.0f}', ha='center', va='bottom', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: MAE Comparison
ax = axes[1, 1]
bars = ax.bar(eval_df['Model'], eval_df['Test MAE'], color=colors, alpha=0.8)
ax.set_title('Mean Absolute Error (Test Set)', fontsize=12, fontweight='bold')
ax.set_ylabel('MAE ($)')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.0f}', ha='center', va='bottom', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../output/02_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Model comparison visualization saved!")

## 9. Future Sales Forecasting (30 Days Ahead)

In [ ]:
# Use the best model for future forecasting
best_model = models[best_model_name]

# Create forecast period (next 30 days)
last_date = daily_sales['date'].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30, freq='D')

# Prepare features for future forecasting
forecast_features = []

for date in future_dates:
    day = date.day
    month = date.month
    quarter = date.quarter
    year = date.year
    day_of_week = date.dayofweek
    day_of_year = date.dayofyear
    is_weekend = 1 if day_of_week >= 5 else 0
    
    # For lag features, use the most recent available values
    lag_1 = df_features['sales'].iloc[-1]  # Last known sales
    lag_7 = df_features['sales'].iloc[-7]  # Sales from 7 days ago
    lag_30 = df_features['sales'].iloc[-30]  # Sales from 30 days ago
    
    rolling_mean_7 = df_features['sales'].iloc[-7:].mean()
    rolling_mean_30 = df_features['sales'].iloc[-30:].mean()
    
    forecast_features.append([day, month, quarter, year, day_of_week, day_of_year,
                              is_weekend, lag_1, lag_7, lag_30, rolling_mean_7, rolling_mean_30])

X_forecast = pd.DataFrame(forecast_features, columns=feature_cols)
forecast_values = best_model.predict(X_forecast)

# Create forecast dataframe
forecast_df = pd.DataFrame({
    'date': future_dates,
    'forecasted_sales': forecast_values
})

print("\n📈 SALES FORECAST - NEXT 30 DAYS")
print("="*60)
print(forecast_df.to_string(index=False))
print(f"\nAverage Forecasted Daily Sales: ${forecast_values.mean():.2f}")
print(f"Total Forecasted Sales (30 days): ${forecast_values.sum():.2f}")

## 10. Forecast Visualization for Business Stakeholders

In [ ]:
# Create comprehensive forecast visualization
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Historical data + Forecast
ax = axes[0]

# Show last 90 days of historical data
recent_data = daily_sales.tail(90)
ax.plot(recent_data['date'], recent_data['sales'], 'o-', linewidth=2.5, 
        markersize=4, label='Historical Sales', color='#2E86AB')

# Plot forecast
ax.plot(forecast_df['date'], forecast_df['forecasted_sales'], 's--', linewidth=2.5,
        markersize=6, label='Forecast', color='#A23B72')

# Add confidence interval (approximate)
test_mae = eval_df[eval_df['Model'] == best_model_name]['Test MAE'].values[0]
upper_bound = forecast_df['forecasted_sales'] + test_mae
lower_bound = forecast_df['forecasted_sales'] - test_mae
ax.fill_between(forecast_df['date'], lower_bound, upper_bound, alpha=0.2, color='#A23B72')

ax.set_title(f'Sales Forecast - Next 30 Days (Model: {best_model_name})', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Sales ($)', fontsize=11)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

# Plot 2: Forecast breakdown by week
ax = axes[1]
forecast_df['week'] = forecast_df['date'].dt.to_period('W')
weekly_forecast = forecast_df.groupby('week')['forecasted_sales'].sum().reset_index()
weekly_forecast['week'] = weekly_forecast['week'].astype(str)

bars = ax.bar(range(len(weekly_forecast)), weekly_forecast['forecasted_sales'], 
              color='#F18F01', alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_title('Weekly Total Sales Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Week', fontsize=11)
ax.set_ylabel('Sales ($)', fontsize=11)
ax.set_xticks(range(len(weekly_forecast)))
ax.set_xticklabels(weekly_forecast['week'], rotation=45)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../output/03_sales_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Forecast visualization saved!")

## 11. Feature Importance Analysis

In [ ]:
# Extract feature importance (works for tree-based models)
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n📊 FEATURE IMPORTANCE ANALYSIS")
    print("="*50)
    print(feature_importance.to_string(index=False))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(feature_importance['feature'], feature_importance['importance'], 
                   color='#2E86AB', alpha=0.8)
    ax.set_title(f'Feature Importance - {best_model_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score', fontsize=11)
    
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2.,
                f'{width:.4f}', ha='left', va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../output/04_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Feature importance visualization saved!")
else:
    print("\n⚠️ Feature importance not available for Linear Regression")

## 12. Business Insights and Recommendations

In [ ]:
# Generate business insights
print("\n" + "="*80)
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("="*80)

# Calculate statistics
historical_avg = daily_sales['sales'].mean()
historical_std = daily_sales['sales'].std()
forecast_avg = forecast_df['forecasted_sales'].mean()
forecast_total = forecast_df['forecasted_sales'].sum()
growth_rate = ((forecast_avg - historical_avg) / historical_avg) * 100

print(f"\n1. FORECAST OVERVIEW")
print(f"   • Historical Average Daily Sales: ${historical_avg:,.2f}")
print(f"   • Forecasted Average Daily Sales (Next 30 Days): ${forecast_avg:,.2f}")
print(f"   • Projected Growth: {growth_rate:+.1f}%")
print(f"   • Total Forecasted Sales (30 days): ${forecast_total:,.2f}")

print(f"\n2. MODEL PERFORMANCE")
print(f"   • Selected Model: {best_model_name}")
print(f"   • Model Accuracy (R² Score): {best_r2:.2%}")
test_mae = eval_df[eval_df['Model'] == best_model_name]['Test MAE'].values[0]
print(f"   • Average Prediction Error: ±${test_mae:,.2f}")
print(f"   • Prediction Confidence: {(1 - (test_mae/historical_avg))*100:.1f}%")

print(f"\n3. KEY TRENDS IDENTIFIED")
max_forecast = forecast_df['forecasted_sales'].max()
min_forecast = forecast_df['forecasted_sales'].min()
max_date = forecast_df[forecast_df['forecasted_sales'] == max_forecast]['date'].values[0]
min_date = forecast_df[forecast_df['forecasted_sales'] == min_forecast]['date'].values[0]
print(f"   • Peak Sales Expected: ${max_forecast:,.2f} on {pd.Timestamp(max_date).strftime('%Y-%m-%d')}")
print(f"   • Lowest Sales Expected: ${min_forecast:,.2f} on {pd.Timestamp(min_date).strftime('%Y-%m-%d')}")
print(f"   • Sales Volatility (Std Dev): ${forecast_df['forecasted_sales'].std():,.2f}")

print(f"\n4. BUSINESS RECOMMENDATIONS")
print(f"   ✓ Inventory Planning: Prepare stock for expected demand peak")
print(f"   ✓ Staffing: Allocate more staff during high-sales periods")
print(f"   ✓ Promotions: Consider promotions during low-sales forecast days")
print(f"   ✓ Cash Flow: Budget for ${forecast_total/30:,.2f} average daily revenue")
print(f"   ✓ Monitoring: Track actual vs. forecasted sales weekly")

## 13. Export Results and Save Model

In [ ]:
import pickle

# Save the best model
model_path = f'../models/{best_model_name.lower().replace(" ", "_")}_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"✅ Model saved: {model_path}")

# Save forecast results
forecast_export = forecast_df.copy()
forecast_export['date'] = forecast_export['date'].astype(str)
forecast_export.to_csv('../output/sales_forecast_30days.csv', index=False)
print(f"✅ Forecast exported: ../output/sales_forecast_30days.csv")

# Save model evaluation results
eval_df.to_csv('../output/model_evaluation_results.csv', index=False)
print(f"✅ Model evaluation saved: ../output/model_evaluation_results.csv")

print("\n" + "="*80)
print("✅ FORECASTING ANALYSIS COMPLETE!")
print("="*80)
print("\nAll results saved to the 'output' folder:")
print("  • 01_time_series_analysis.png")
print("  • 02_model_comparison.png")
print("  • 03_sales_forecast.png")
print("  • 04_feature_importance.png")
print("  • sales_forecast_30days.csv")
print("  • model_evaluation_results.csv")